# Linear algebra computational companion: solving spatial estimation problems

**Foundation:** [SM-FND-002 — Linear algebra and matrix methods](../foundations/linear-algebra/README.md)  
**Level:** introductory to intermediate  
**Dependencies:** NumPy  
**Data:** entirely synthetic; no real survey or geographic observations are used.

This lab connects the algebra in Modules 05–10 to computational choices that recur in coordinate estimation, survey adjustment, regression and uncertainty propagation. We use one small synthetic observation model to compare direct solution, least squares, QR, Cholesky whitening, SVD and covariance propagation.

The aim is not to teach a software API by itself. At each step, ask: **what mathematical problem is being solved, what assumptions make the operation valid, and what evidence would still be needed before trusting a real-world result?**

## 1. A linear observation model

Suppose a set of synthetic observations follows

\[
\mathbf y = \mathbf A\boldsymbol\beta + \boldsymbol\varepsilon,
\]

where the first column of \(\mathbf A\) is an intercept and the second is a one-dimensional coordinate. The unknown parameter vector contains an intercept and a slope.

We deliberately use a small, transparent dataset. The numbers are illustrative and do not represent a real place, instrument or survey.

In [ ]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)

x = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
A = np.column_stack([np.ones_like(x), x])
y = np.array([10.2, 12.1, 13.8, 16.2, 17.9, 20.1])

beta_true_for_simulation = np.array([10.0, 2.0])
beta_ols, *_ = np.linalg.lstsq(A, y, rcond=None)
residuals = y - A @ beta_ols

print("Design matrix shape:", A.shape)
print("Estimated [intercept, slope]:", beta_ols)
print("Residual sum of squares:", residuals @ residuals)
assert A.shape == (6, 2)
assert np.linalg.matrix_rank(A) == 2
assert np.allclose(A @ beta_ols + residuals, y)

## 2. Normal equations versus QR

The ordinary least-squares estimate minimises \(\|\mathbf A\beta-\mathbf y\|_2^2\). If the columns of \(A\) have full rank, the normal-equation expression is

\[
\hat\beta=(A^TA)^{-1}A^Ty.
\]

In computation, do not form the inverse explicitly. Even solving the normal equations squares the 2-norm condition number: \(\kappa(A^TA)=\kappa(A)^2\) for full-column-rank \(A\). QR works with an orthogonal-triangular factorisation and is generally a more stable least-squares route.

In [ ]:
# Solve the normal equations without explicitly forming an inverse.
normal_matrix = A.T @ A
normal_rhs = A.T @ y
beta_normal = np.linalg.solve(normal_matrix, normal_rhs)

# Reduced QR: A = Q R, then solve R beta = Q.T y.
Q, R = np.linalg.qr(A, mode="reduced")
beta_qr = np.linalg.solve(R, Q.T @ y)

print("Normal-equation estimate:", beta_normal)
print("QR estimate:", beta_qr)
print("Difference:", beta_normal - beta_qr)
print("Condition number of A:", np.linalg.cond(A))
print("Condition number of A.T @ A:", np.linalg.cond(normal_matrix))

assert np.allclose(beta_normal, beta_qr, rtol=1e-10, atol=1e-10)
assert np.allclose(beta_qr, beta_ols, rtol=1e-10, atol=1e-10)

## 3. Cholesky factorisation and correlated observations

Ordinary least squares treats observation errors as having equal variance and no correlation. Suppose instead that the covariance matrix is

\[
\Sigma = D + \tau^2\mathbf 1\mathbf 1^T,
\]

where \(D\) contains observation-specific variances and the second term represents a shared additive error component. This is a simple positive-definite covariance model when all diagonal variances are positive and \(\tau^2\geq0\).

Generalised least squares minimises the covariance-weighted residual quadratic form. If \(\Sigma=LL^T\), premultiplying by \(L^{-1}\) whitens the system. We can then solve an ordinary least-squares problem in whitened coordinates. We use triangular solves rather than explicitly computing \(L^{-1}\) or \(\Sigma^{-1}\).

In [ ]:
sigma_individual = np.array([0.20, 0.25, 0.22, 0.30, 0.24, 0.28])
shared_sigma = 0.15
Sigma = np.diag(sigma_individual**2) + shared_sigma**2 * np.ones((len(x), len(x)))

# Cholesky requires a symmetric positive-definite covariance matrix.
L = np.linalg.cholesky(Sigma)
A_white = np.linalg.solve(L, A)
y_white = np.linalg.solve(L, y)

Qw, Rw = np.linalg.qr(A_white, mode="reduced")
beta_gls = np.linalg.solve(Rw, Qw.T @ y_white)

# Independent algebraic check of the GLS normal equations.
Sigma_inv_A = np.linalg.solve(Sigma, A)
Sigma_inv_y = np.linalg.solve(Sigma, y)
beta_gls_check = np.linalg.solve(A.T @ Sigma_inv_A, A.T @ Sigma_inv_y)

print("GLS estimate:", beta_gls)
print("GLS check:", beta_gls_check)
print("OLS estimate:", beta_ols)
print("Cholesky reconstruction error:", np.linalg.norm(Sigma - L @ L.T))

assert np.allclose(Sigma, Sigma.T)
assert np.all(np.linalg.eigvalsh(Sigma) > 0)
assert np.allclose(Sigma, L @ L.T)
assert np.allclose(beta_gls, beta_gls_check, rtol=1e-10, atol=1e-10)

## 4. SVD and rank diagnosis

The singular-value decomposition is

\[
A=U\,\operatorname{diag}(s_i)\,V^T.
\]

Singular values describe how strongly the matrix acts along orthogonal directions. A very small singular value can indicate a nearly unidentifiable parameter combination. Numerical rank depends on a tolerance; it is not an absolute metaphysical property of the dataset.

We create a deliberately ill-conditioned design by making the second coordinate column almost a constant multiple of the first. The example illustrates numerical sensitivity, not a recommended way to repair a design.

In [ ]:
x_near = 1.0 + np.array([0.0, 1e-8, 2e-8, 3e-8, 4e-8, 5e-8])
A_near = np.column_stack([np.ones_like(x_near), x_near])
U, singular_values, Vt = np.linalg.svd(A_near, full_matrices=False)

default_tolerance = np.finfo(float).eps * max(A_near.shape) * singular_values[0]
numerical_rank = int(np.sum(singular_values > default_tolerance))

print("Singular values:", singular_values)
print("Condition number:", np.linalg.cond(A_near))
print("Numerical rank at default tolerance:", numerical_rank)
print("Right singular vectors (rows of V.T):\n", Vt)

assert singular_values[0] >= singular_values[1]
assert numerical_rank == 2

## 5. Propagating covariance through a linear transformation

If a vector \(x\) has covariance \(\Sigma_x\), and a deterministic linear transformation produces \(z=Jx\), then

\[
\Sigma_z=J\Sigma_xJ^T.
\]

This is exact for a linear transformation. For a nonlinear function, the same expression is a first-order approximation when \(J\) is the Jacobian evaluated at an appropriate point.

Here we use a simple 2D rotation-like linear map and an illustrative input covariance. The matrix operation is not a substitute for defining the physical coordinate frames, units, datum or transformation model.

In [ ]:
angle = np.deg2rad(30.0)
J = np.array([
    [np.cos(angle), -np.sin(angle)],
    [np.sin(angle),  np.cos(angle)],
])
Sigma_x = np.array([[0.04, 0.01], [0.01, 0.09]])
Sigma_z = J @ Sigma_x @ J.T

print("Input covariance:\n", Sigma_x)
print("Output covariance:\n", Sigma_z)
print("Input eigenvalues:", np.linalg.eigvalsh(Sigma_x))
print("Output eigenvalues:", np.linalg.eigvalsh(Sigma_z))

assert np.allclose(Sigma_z, Sigma_z.T)
assert np.all(np.linalg.eigvalsh(Sigma_z) >= -1e-12)
assert np.allclose(np.linalg.eigvalsh(Sigma_x), np.linalg.eigvalsh(Sigma_z))

## 6. Interpretation and limits

The computational choices answer different questions:

| Tool | Appropriate role | Important caution |
|---|---|---|
| Direct linear solve | A square, nonsingular system | Does not solve an overdetermined least-squares problem by itself |
| Normal equations | Small, well-conditioned teaching examples | Squaring the condition number can amplify numerical problems |
| QR | Full-rank least squares | Rank deficiency still requires explicit diagnosis and a suitable policy |
| Cholesky | Positive-definite covariance/normal matrix factorisation | Fails for matrices that are not positive definite; do not silently repair a bad covariance |
| SVD | Rank diagnosis and minimum-norm least squares | Rank depends on scale and tolerance; interpretation needs context |
| Covariance propagation | Linear uncertainty transformation | Correct algebra cannot rescue an invalid input covariance or an incorrect physical model |

### Questions to investigate

1. Change the shared error component \(\tau\). How does the GLS estimate respond, and why does a common-mode component affect the intercept and slope differently?
2. Rescale the coordinate column by a factor of 1,000. What happens to the condition number, and what does this say about parameter scaling?
3. Remove one observation or make all coordinate values identical. Which parameters become unidentifiable?
4. Replace the covariance matrix with one that is not positive definite. Why does Cholesky fail, and what should be investigated before choosing another factorisation?
5. In a real survey or spatial model, what evidence is needed to justify the observation model, covariance structure, coordinate reference and stochastic assumptions?

### What this lab does not establish

All values are synthetic. Successful execution verifies only that these particular calculations run and satisfy the included numerical assertions. It does not validate a real dataset, a survey network, a covariance model, a coordinate transformation or a production estimation system.

## References and further study

- [Linear algebra and matrix methods](../foundations/linear-algebra/README.md)
- [Sequence audit and outstanding review work](../foundations/linear-algebra/sequence-audit.md)
- [Covariance and correlated observations](../foundations/measurement-science/covariance-correlated-observations-and-gls.md)
- [NumPy linear algebra reference](https://numpy.org/doc/stable/reference/routines.linalg.html)

The notebook is an original teaching companion. Consult the linked foundation modules for derivations and the independent review status; this notebook is not a substitute for either.